# Project 1 — Multi-Modal Image Search: Colab Prototype

**Purpose**: prototype CLIP-based text/image search on a small sample of photos using a
free Colab GPU. This notebook is for experimentation only — it does not run against
your full private library, and its code is not the deployment target.

Finalized logic from this notebook should be manually ported into
`../src/embed.py`, `../src/ingest.py`, `../src/index_store.py`, and `../src/search.py`
for the local Streamlit app (`../src/app.py`).

**Workflow**: mount Drive → install deps → load a small sample folder (10–50 photos)
→ embed with `clip-ViT-B-32` → build a FAISS index → try text-to-image and
image-to-image search → inspect results as a thumbnail grid.


## 1. Environment and Dependency Setup

In [ ]:
# Runs only on Colab; harmless no-op locally.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q sentence-transformers faiss-cpu pillow

print(f"IN_COLAB={IN_COLAB}")

In [ ]:
import numpy as np
import faiss
from pathlib import Path
from PIL import Image
from sentence_transformers import SentenceTransformer

MODEL_NAME = "clip-ViT-B-32"
model = SentenceTransformer(MODEL_NAME)
print(f"Loaded {MODEL_NAME}, embedding dim = {model.get_sentence_embedding_dimension()}")

## 2. Project Structure and Configuration

If running on Colab, mount Google Drive and point `SAMPLE_DIR` at a small folder
(10–50 photos) you've uploaded there. Never point this at your full private library —
Colab is for prototyping only.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SAMPLE_DIR = Path("/content/drive/MyDrive/photo_search_sample")
else:
    # Local dry-run: point this at a small local sample folder instead.
    SAMPLE_DIR = Path("./sample_photos")

INDEX_PATH = Path("./data/prototype_index.faiss")
INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
TOP_K = 5

assert SAMPLE_DIR.exists(), f"Sample folder not found: {SAMPLE_DIR}"
print(f"SAMPLE_DIR = {SAMPLE_DIR.resolve()}")

## 3. Data Loading and Validation

Scan the sample folder, confirm each file is a readable image, and report basic
counts before spending time on embeddings.

In [ ]:
candidate_paths = [p for p in SAMPLE_DIR.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS]

valid_paths = []
for p in candidate_paths:
    try:
        with Image.open(p) as img:
            img.verify()
        valid_paths.append(p)
    except Exception as e:
        print(f"Skipping unreadable file {p.name}: {e}")

print(f"Found {len(candidate_paths)} candidate files, {len(valid_paths)} valid images")
assert len(valid_paths) > 0, "No valid images found in SAMPLE_DIR"

## 4. Baseline Processing Pipeline

Reusable functions to embed a single image, embed a batch, and encode a text query —
these map directly to `../src/embed.py`.

In [ ]:
def encode_image(path):
    image = Image.open(path).convert("RGB")
    return model.encode(image, convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)


def encode_text(text):
    return model.encode(text, convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)


def embed_batch(paths):
    vectors = np.stack([encode_image(p) for p in paths])
    return vectors


sample_paths = valid_paths[:10]
sample_vectors = embed_batch(sample_paths)
print(f"Embedded {len(sample_paths)} sample images -> shape {sample_vectors.shape}")

## 5. Initial Model or Core Logic Implementation

Embed the full validated sample, build a FAISS index (cosine similarity via inner
product on normalized vectors), and implement `search()` for both text and image
queries — mirrors `../src/index_store.py` and `../src/search.py`.

In [ ]:
all_vectors = embed_batch(valid_paths)
dim = all_vectors.shape[1]

index = faiss.IndexIDMap(faiss.IndexFlatIP(dim))
ids = np.arange(len(valid_paths), dtype=np.int64)
index.add_with_ids(all_vectors, ids)
faiss.write_index(index, str(INDEX_PATH))

id_to_path = {i: p for i, p in zip(ids, valid_paths)}
print(f"Indexed {index.ntotal} vectors -> {INDEX_PATH}")


def search(query_vector, top_k=TOP_K):
    query_vector = query_vector.reshape(1, -1).astype(np.float32)
    scores, result_ids = index.search(query_vector, top_k)
    return [(id_to_path[i], float(s)) for i, s in zip(result_ids[0], scores[0]) if i != -1]

## 6. Run, Evaluate, and Save Outputs

Try a text query and an image query, render the top matches as a thumbnail grid, and
save results to `results.json` for reference.

In [ ]:
import matplotlib.pyplot as plt

TEXT_QUERY = "a dog on the beach"  # TODO: try your own descriptions

text_results = search(encode_text(TEXT_QUERY))
print(f"Text query: {TEXT_QUERY!r}")
for path, score in text_results:
    print(f"  {score:.3f}  {path.name}")

fig, axes = plt.subplots(1, len(text_results), figsize=(4 * len(text_results), 4))
if len(text_results) == 1:
    axes = [axes]
for ax, (path, score) in zip(axes, text_results):
    ax.imshow(Image.open(path))
    ax.set_title(f"{score:.3f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
REFERENCE_IMAGE = valid_paths[0]  # TODO: pick a different reference photo

image_results = search(encode_image(REFERENCE_IMAGE))
print(f"Image query: {REFERENCE_IMAGE.name}")
for path, score in image_results:
    print(f"  {score:.3f}  {path.name}")

fig, axes = plt.subplots(1, len(image_results), figsize=(4 * len(image_results), 4))
if len(image_results) == 1:
    axes = [axes]
for ax, (path, score) in zip(axes, image_results):
    ax.imshow(Image.open(path))
    ax.set_title(f"{score:.3f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import json

results_path = Path("./results.json")
results_path.write_text(json.dumps({
    "text_query": TEXT_QUERY,
    "text_results": [{"path": str(p), "score": s} for p, s in text_results],
    "reference_image": str(REFERENCE_IMAGE),
    "image_results": [{"path": str(p), "score": s} for p, s in image_results],
}, indent=2))
print(f"Saved results -> {results_path.resolve()}")